<a href="https://colab.research.google.com/github/maya-hoff/CS143-Portfolio/blob/main/Copy_of_F4_2_PyTorchEmbeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 195: Natural Language Processing
## PyTorch Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F4_2_PyTorchEmbeddings.ipynb)


## References

[SLP: Embeddings, Chapter 5 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/5.pdf)

[Word2Vec Tutorial - The Skip-Gram Model by Chris McCormick](http://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/)

[Word2Vec - Negative Sampling made easy by Munesh Lakhey](https://medium.com/@mnshonco/word2vec-negative-sampling-made-easy-9a587cb4695f)

[PyTorch `nn.Embedding` documentation](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

[PyTorch `torch.nn.functional.one_hot` documentation](https://pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)


In [1]:
#import sys
#!{sys.executable} -m pip install datasets torch scikit-learn transformers tokenizers

## Reorganization of where we left off from last time

In [2]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import random
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim

def train_tokenizer(sentences, vocabulary_size=200):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocabulary_size)

    tokenizer.train_from_iterator(sentences, trainer)
    return tokenizer


def make_skipgrams(sequence, vocabulary_size, window_size=3):
    couples = []
    labels = []

    for i in range(len(sequence)):
        target = sequence[i]
        left = max(0, i - window_size)
        right = min(len(sequence), i + window_size + 1)

        for j in range(left, right):
            if i != j:
                context = sequence[j]

                # positive pair
                couples.append((target, context))
                labels.append(1)

                # generate a random negative pair (in real life, you might want to generate several negative pairs per positive pair)
                negative_context = random.randint(1, vocabulary_size - 1)
                # TODO: we should probably check to make sure this isn't actually a positive pair
                couples.append((target, negative_context))
                labels.append(0)

    return [couples, labels]

def make_skipgrams_batch(tokenized_sentences, vocabulary_size, window_size=3):
    all_couples = []
    all_labels = []
    for tokenized_sentence in tokenized_sentences:
        couples, labels = make_skipgrams(tokenized_sentence, vocabulary_size, window_size)
        all_couples.extend(couples)
        all_labels.extend(labels)

    return [all_couples, all_labels]


def prepare_one_hot_inputs(couples, labels, vocabulary_size):
    inputs = []
    for target_word, context_word in couples:
        target_one_hot = F.one_hot(torch.tensor(target_word), num_classes=vocabulary_size)
        context_one_hot = F.one_hot(torch.tensor(context_word), num_classes=vocabulary_size)
        inputs.append(torch.cat([target_one_hot, context_one_hot]))

    inputs_array = torch.stack(inputs).float()
    labels_array = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return inputs_array, labels_array


def train_embedding_model(inputs_array, labels_array, vocabulary_size):

    embedding_model = nn.Sequential(
        nn.Linear(vocabulary_size * 2, 50),
        nn.ReLU(),
        nn.Linear(50, 1)
    )

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function
    optimizer = optim.Adam(embedding_model.parameters(), lr=0.0001)

    for epoch in range(20000):
        logits = embedding_model(inputs_array)
        loss = loss_fn(logits, labels_array)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 1000 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids
    first_layer_weights = embedding_model[0].weight.detach()
    return first_layer_weights[:, word_ids[0]] # use first subword token for this demo



In [3]:
sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

tokenizer = train_tokenizer(sentences, vocabulary_size=200)

tokenized_sentences = []
for sentence in sentences:
    ids = tokenizer.encode(sentence).ids
    tokens = tokenizer.encode(sentence).tokens
    tokenized_sentences.append(ids)

print("Here's an example of some tokens:", tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)

inputs_array, labels_array = prepare_one_hot_inputs(couples, labels, tokenizer.get_vocab_size())

embedding_model = train_embedding_model(inputs_array, labels_array, vocabulary_size=tokenizer.get_vocab_size())


Here's an example of some tokens: ['coral', 'reefs', 'are', 'in', 'the', 'ocean']
epoch 1000, loss=0.3864
epoch 2000, loss=0.2489
epoch 3000, loss=0.1883
epoch 4000, loss=0.1435
epoch 5000, loss=0.1075
epoch 6000, loss=0.0808
epoch 7000, loss=0.0633
epoch 8000, loss=0.0525
epoch 9000, loss=0.0461
epoch 10000, loss=0.0424
epoch 11000, loss=0.0402
epoch 12000, loss=0.0390
epoch 13000, loss=0.0382
epoch 14000, loss=0.0378
epoch 15000, loss=0.0376
epoch 16000, loss=0.0374
epoch 17000, loss=0.0373
epoch 18000, loss=0.0373
epoch 19000, loss=0.0372
epoch 20000, loss=0.0372


In [4]:

cats_embedding = get_embedding("cats", tokenizer, embedding_model)
dogs_embedding = get_embedding("dogs", tokenizer, embedding_model)
ocean_embedding = get_embedding("ocean", tokenizer, embedding_model)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)


tensor([-0.5119,  0.8449,  0.7306,  0.7934,  1.0758, -0.1052,  0.5699,  0.7868,
         0.3938, -0.9885,  0.0539, -0.7187, -0.8240,  0.8399,  0.5041,  0.1892,
         0.8099,  0.8567,  0.1939,  0.9958, -0.9812, -0.0098,  0.4586,  0.8246,
        -0.7709,  0.8627,  0.9322,  0.0460, -1.0201,  0.7860,  0.9220, -0.5073,
        -0.2591, -0.9634,  0.8626,  0.3235,  0.8298, -0.9659,  0.7530,  0.9211,
         0.7162, -0.3185, -0.3481, -0.6153,  0.9473,  0.8330, -0.0079,  0.9157,
        -0.9075,  0.9500])
tensor([ 0.9286,  1.1235,  0.7433,  0.8844,  0.8001,  0.9032, -0.8927,  0.8981,
         0.3939, -1.0030,  0.5575,  0.9644,  0.9357,  0.8420, -0.5496, -0.9730,
         0.8793,  0.3862,  0.8039,  1.2075,  1.3001,  0.7742,  1.0319,  0.8250,
        -0.9134,  0.8627, -0.6571,  1.2892,  0.4834,  0.8074,  0.9337, -0.8427,
         0.8054, -0.2704,  0.7768, -1.0089,  0.8520,  0.5833,  1.1651,  0.8261,
         0.8007, -0.0189,  0.9177, -1.0021, -0.9455, -0.5983, -0.5443,  1.1861,
         0.68

## Dataset for today

AG News dataset
* short news articles
* four classes: World, Sports, Business, Sci/Tech

https://huggingface.co/datasets/fancyzhx/ag_news


In [5]:
from datasets import load_dataset
data = load_dataset("ag_news")

print(data["train"]["text"][0])

# 0 is World
# 1 is Sports
# 2 is Business
# 3 is Sci/Tech
print(data["train"]["label"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2


## Group Exercise

Try creating word embeddings for the AG News dataset. *Note that you will only be able to do a very small subset with this code. Start with 10 examples and work your way up*

* How big of a vocabulary did you need to use to get reasonable tokens for this data?
* Once you use more examples, you'll start to notice that you're going to run out of memory and the kernel will crash. What do you think the issue is? Can you think of a way to do the same idea but keep less data in memory at any given time?



## The PyTorch Embedding Layer
PyTorch provides an `nn.Embedding` layer, which is a linear layer with a lookup table. It allows you to input a token id and look up a row vector (akin to the linear node associated with that id).

It's mathematically equivalent to the one-hot encoding + `nn.Linear` layer we saw before, but it much more memory efficient - we don't have to waste space storing all of those 0s.

Let's try the same experiment as before but using the `nn.Embedding` layer.


In [6]:
def train_embedding_model_v2(couples, labels, vocabulary_size):

    embedding_model = nn.Embedding(vocabulary_size, 50)
    skipgram_classifier = nn.Linear(2*50, 1) # target emb + context emb

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function
    optimizer = optim.Adam(list(embedding_model.parameters()) + list(skipgram_classifier.parameters()), lr=0.0003) # concatenate the parameters for the embedding model and skipgram classifier

    pair_ids = torch.tensor(couples, dtype=torch.long)                # [N, 2]
    labels_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # [N, 1]

    for epoch in range(5000):
        target_ids = pair_ids[:, 0] # [N]
        context_ids = pair_ids[:, 1] # [N]

        target_embeddings = embedding_model(target_ids)  # [N, 50]
        context_embeddings = embedding_model(context_ids)  # [N, 50]

        target_context_together = torch.cat([target_embeddings, context_embeddings], dim=1)  # [N, 100]

        logits = skipgram_classifier(target_context_together)
        loss = loss_fn(logits, labels_tensor )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 500 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding_v2(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids[0] # use first subword token for this demo
    word_weights = embedding_model.weight.detach()
    return word_weights[word_ids]

sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

tokenizer = train_tokenizer(sentences, vocabulary_size=200)

tokenized_sentences = []
for sentence in sentences:
    ids = tokenizer.encode(sentence).ids
    tokens = tokenizer.encode(sentence).tokens
    tokenized_sentences.append(ids)

print(tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)
embedding_model_v2 = train_embedding_model_v2(couples, labels, vocabulary_size=tokenizer.get_vocab_size())

['coral', 'reefs', 'are', 'in', 'the', 'ocean']
epoch 500, loss=0.3912
epoch 1000, loss=0.2731
epoch 1500, loss=0.2528
epoch 2000, loss=0.2473
epoch 2500, loss=0.2451
epoch 3000, loss=0.2441
epoch 3500, loss=0.2435
epoch 4000, loss=0.2432
epoch 4500, loss=0.2430
epoch 5000, loss=0.2429


In [7]:

cats_embedding = get_embedding_v2("cats", tokenizer, embedding_model_v2)
dogs_embedding = get_embedding_v2("dogs", tokenizer, embedding_model_v2)
ocean_embedding = get_embedding_v2("ocean", tokenizer, embedding_model_v2)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)

tensor([-0.2691,  0.9073,  0.5678, -0.8992, -0.3422, -1.0579,  1.5106, -0.7537,
        -0.5751, -0.2063, -1.3671,  0.1217, -0.5128, -0.2125, -0.1843, -0.7034,
        -0.9790, -2.1649, -0.6204,  0.6253, -0.8918,  0.5249, -0.6320,  0.3200,
        -0.1112, -0.0153, -0.8297,  0.5482, -0.4017,  0.2919,  1.9205,  0.0628,
        -0.2814, -0.0593,  0.6118, -0.1656,  1.9346,  0.3552,  0.5607,  0.7454,
         1.7394, -0.1745,  0.5108,  0.2585,  1.3857, -1.1516,  0.6599,  0.2105,
         0.4155, -1.2797])
tensor([-1.1537, -1.1364,  0.2490,  1.1048,  0.4585,  0.0050,  1.6831, -0.5284,
         0.7114,  0.4571,  0.5005,  0.9690, -0.7006,  1.0654, -0.2438,  0.0982,
        -0.5544, -0.1806, -0.0900, -0.5328, -0.2161, -1.0204,  0.4281,  0.5791,
        -0.9588, -0.5856, -1.5589, -0.4769,  0.0330,  0.8115,  0.8584,  0.5519,
         0.3799,  0.5576,  1.0741,  1.8417, -0.3208, -0.5611, -0.6699,  0.2499,
        -0.7660, -0.1586,  3.2224, -1.4466,  0.1646, -1.0430, -0.0428, -0.3432,
        -1.50

### Caveat

With such a small dataset, we're not actually getting good embeddings yet.

## Applied Exploration

Create word embeddings for a larger portion of the AG News dataset, say 5000 texts

Show some example word embeddings for some words that appear in the dataset (*cats* and *dogs* may not be good examples for this one)

Describe your results and reflect on them
* How big of a vocabulary did you need to use?
* What learning rate and number of training epochs do you think are appropriate? Why?
* How could you go about figuring out if these embeddings are useful?

## Adding an Embedding layer to your model for other learning tasks

First, let's prepare the data
* We could use the same kind of tokenizer, or just use an existing one - let's go back to doing it with a pretrained Hugging Face tokenizer


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

data = load_dataset("ag_news")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]
train_labels = data["train"]["label"]
test_labels = data["test"]["label"]

hf_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
hf_tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # for some reason this tokenizer doesn't have a pad token by default, but we need one to be able to batch our inputs together, so we'll just add a new special token for padding

tokenized_train_texts = hf_tokenizer(list(train_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")
tokenized_test_texts = hf_tokenizer(list(test_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")

X_train = tokenized_train_texts["input_ids"]
X_test = tokenized_test_texts["input_ids"]

y_train = torch.tensor(np.array(train_labels), dtype=torch.long)
y_test = torch.tensor(np.array(test_labels), dtype=torch.long)


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

## Model with an Embedding layer

We'll set up the embedding layer and sequential model as before

The optimizer needs to have a concatenation of the parameters for both parts

In [9]:
embedding = nn.Embedding(len(hf_tokenizer), 50) # len(hf_tokenizer) is the vocab size including the new padding token
classifier = nn.Sequential(
    nn.Linear(50, 100),
    nn.ReLU(),
    nn.Linear(100, 4)
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(embedding.parameters()) + list(classifier.parameters()), lr=0.01)

## Training Loop with Embedding layer

The input goes into the embedding layer
* returns an embedding for each words, so we aggregate them with the *mean* to get an embedding for the whole training example

In [10]:


for epoch in range(100):
    optimizer.zero_grad()

    emb = embedding(X_train) # [batch_size, seq_len, embedding_dim]
    pooled_emb = emb.mean(dim=1) # simple way to get a single vector [batch_size, embedding_dim] for the whole sequence - just average the token embeddings
    logits = classifier(pooled_emb)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")



Epoch 1, loss = 1.3883
Epoch 11, loss = 1.3468
Epoch 21, loss = 1.1090
Epoch 31, loss = 0.6973
Epoch 41, loss = 0.4103
Epoch 51, loss = 0.2964
Epoch 61, loss = 0.2330
Epoch 71, loss = 0.1874
Epoch 81, loss = 0.1550
Epoch 91, loss = 0.1299


## Evaluating works similarly

In [ ]:
with torch.no_grad():
    emb = embedding(X_test)
    pooled_emb = emb.mean(dim=1)
    logits = classifier(pooled_emb)
    predicted_labels = logits.argmax(dim=1)
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 0.902236819267273


## Applied Exploration

Perform a text classification experiment with another classification data set
* Try different embedding vector lengths (other than just 50 as we did here)
* Experiment with different neural network structures, learning rates, and number of epochs

Report your results and reflect on them
* Describe your dataset
* Describe what you did
* Report the results you observed
* Discuss any interesting insights

In [18]:
data = load_dataset("sms_spam")

split_data = data["train"].train_test_split(test_size=0.2, seed=42)

train_texts = split_data["train"]["sms"]
test_texts = split_data["test"]["sms"]

train_labels = split_data["train"]["label"]
test_labels = split_data["test"]["label"]

hf_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
hf_tokenizer.add_special_tokens({'pad_token': '[PAD]'})

tokenized_train_texts = hf_tokenizer(list(train_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")
tokenized_test_texts = hf_tokenizer(list(test_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")

X_train = tokenized_train_texts["input_ids"]
X_test = tokenized_test_texts["input_ids"]

y_train = torch.tensor(np.array(train_labels), dtype=torch.long)
y_test = torch.tensor(np.array(test_labels), dtype=torch.long)

embed_size = 100

embedding = nn.Embedding(len(hf_tokenizer), embed_size)
classifier = nn.Sequential(
    nn.Linear(embed_size, 100),
    nn.ReLU(),
    nn.Linear(100, 2)
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(embedding.parameters()) + list(classifier.parameters()), lr=0.001)

for epoch in range(50):
    optimizer.zero_grad()

    emb = embedding(X_train)
    pooled_emb = emb.mean(dim=1)
    logits = classifier(pooled_emb)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")

with torch.no_grad():
    emb = embedding(X_test)
    pooled_emb = emb.mean(dim=1)
    logits = classifier(pooled_emb)
    predicted_labels = logits.argmax(dim=1)
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())

Epoch 1, loss = 0.6149
Epoch 11, loss = 0.3256
Epoch 21, loss = 0.3269
Epoch 31, loss = 0.3135
Epoch 41, loss = 0.3012
Accuracy: 0.8663676977157593


Dataset used: https://huggingface.co/datasets/ucirvine/sms_spam
# Results
embedding | learning rate | epochs | accuracy
--|--|--|--|
25 | 0.01 | 25 | 87%
50 | 0.01 | 25 | 87%
100 | 0.001 | 50 | 87%

1. The dataset used is one that contains SMS messages and classifies them as spam (1) or not spam (0).
2. I used the SMS Spam dataset to train a neural network for text classification and experimented with different embedding sizes, learning rates, and numbers of epochs.
3. Changing the embedding size and training settings didn't really affect the model's accuracy. This suggests the simple feedforward network may not capture enough information from the text. Using a more advanced model could potentially improve performance.